<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/scenario-portfolio-lab/notebooks/06_portfolio_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El botón abre esta notebook en Google Colab. Allí ejecuta primero la siguiente celda de preparación para obtener el código y los reportes públicos; no descarga datos de mercado ni solicita credenciales.

# 07. Evaluación temporal de las reglas de cartera

## ¿Qué habría ocurrido al construir y mantener carteras con estas reglas, utilizando únicamente la información admisible en cada decisión?

En 06 construimos una cartera candidata de mínimo CVaR en un único corte histórico. Ese resultado describe su comportamiento bajo los escenarios utilizados para optimizarla, no su desempeño posterior.

En 07 reconstruiremos decisiones sucesivas y mediremos lo que ocurre después de cada una. Compararemos tres reglas: pesos iguales, mínimo CVaR con probabilidades exponenciales y mínimo CVaR condicionado por VIX.

Conservaremos los ocho ETFs y las restricciones de 06: cartera plenamente invertida, sin posiciones cortas, margen ni efectivo, y un tope de 15% por instrumento. Los escenarios de riesgo serán inicialmente de una sesión; el calendario de rebalanceo y la convención de ejecución se definirán por separado.

El ejercicio será retrospectivo. El universo se seleccionó después de observar la historia y la muestra ya fue explorada, por lo que no representa un *holdout* intacto ni resultados de operación real.

**Siguiente pregunta:** ¿Cuándo calculamos los pesos, cuándo podemos ejecutarlos y desde qué momento les atribuimos resultados?

## 2. ¿Cuándo decidimos y desde cuándo cuentan los resultados?

Todos los desplazamientos de esta convención corresponden a sesiones del calendario publicado, no a días naturales. Separamos la información que sirve para decidir de los retornos que atribuimos después a la decisión.

| Momento | Información o acción |
| --- | --- |
| Hasta el cierre de $t-1$ | Últimos retornos utilizados para ajustar el modelo. |
| Antes de ejecutar en $t$ | Calculamos los pesos con esa historia y el VIX de $t-1$. |
| Al cierre de $t$ | Ejecutamos hipotéticamente el rebalanceo. |
| Del cierre de $t$ al cierre de $t+1$ | Primer retorno atribuible a los pesos nuevos. |

Ejecutar al cierre es un supuesto del backtest, no una garantía de obtener ese precio en una operación real.

El retorno fechado $t$ pertenece a los pesos anteriores. En la primera inversión, la evaluación comienza después de ejecutar la cartera; todavía no calculamos resultados del periodo anterior a esa inversión.

No usamos el retorno ni el VIX de $t$ para decidir los pesos ejecutados al cierre de $t$.

Consideramos disponible el VIX de $t-1$ antes de ejecutar en $t$, siguiendo la evidencia aportada por el proveedor: normalmente estaría disponible hacia las 22:00 US/Eastern. Esta es una convención retrospectiva basada en esa comunicación, no una disponibilidad histórica verificada para cada fecha ni una garantía de publicación.

El VIX tiene dos usos distintos. En la muestra histórica de ajuste, cada retorno fechado $x$ conserva su asociación con el VIX de $x-2$, según la convención de 04/05. El VIX de $t-1$ identifica el estado conocido para la decisión actual. No cambiaremos la alineación histórica para hacer coincidir ambas fechas.

Las tres reglas —pesos iguales, mínimo CVaR exponencial y mínimo CVaR condicionado por VIX— compartirán la misma convención de decisión, ejecución y atribución de resultados.

Como ejemplo conceptual, considere tres sesiones consecutivas. En la sesión anterior termina la información utilizada; en la sesión de ejecución aplicamos los nuevos pesos al cierre; en la sesión siguiente observamos el primer retorno de los nuevos pesos. Este ejemplo no presupone precios, pesos ni rendimientos concretos.

**Siguiente pregunta:** ¿En qué sesiones rebalancearemos y cuánta historia exigiremos antes de invertir?

## 3. ¿Cuándo rebalanceamos y cuánta historia necesitamos?

### Frecuencia mensual

Para las tres reglas, el rebalanceo se ejecutará hipotéticamente al cierre de la última sesión publicada de cada mes calendario completo. La decisión utiliza información hasta la sesión anterior a la ejecución, según la convención temporal anterior. Identificaremos esas fechas con el calendario publicado, sin suponer que todos los meses terminan en un día hábil. El último día disponible del corte no se tratará automáticamente como cierre mensual: octubre de 2026 es parcial.

La frecuencia mensual es una decisión operativa inicial, no una frecuencia demostrada como óptima.

### Historia mínima

Exigiremos al menos 504 observaciones históricas alineadas y admisibles para el modelo condicionado por VIX. El mínimo CVaR exponencial utilizará exactamente esas mismas fechas de ajuste. Las 504 observaciones son filas conjuntas de retornos y estado histórico asociado, no días naturales.

La ventana será expansiva: comienza en la primera observación alineada disponible y crece en cada rebalanceo. La primera inversión ocurrirá en el primer cierre mensual que cumpla este requisito; todavía no calculamos cuál es esa fecha. Las 504 observaciones son un requisito de diseño, no una garantía de precisión.

### Parámetros fijos

Mantendremos una semivida exponencial de 1,102 sesiones, CVaR al 95% y un horizonte de riesgo de una sesión, junto con los ocho ETFs y las restricciones de 06. En cada rebalanceo condicionado reconstruiremos los cortes de VIX y los objetivos de *entropy pooling* con la muestra admisible de esa fecha. No reutilizaremos cortes ni probabilidades ajustadas con toda la historia de 04, ni buscaremos parámetros retrospectivamente para mejorar resultados.

Optimizar riesgo de una sesión y rebalancear mensualmente son decisiones distintas: no estamos minimizando directamente el CVaR mensual.

### Entre rebalanceos

No restauraremos diariamente los pesos objetivo. Los pesos evolucionarán con los retornos de los instrumentos. Pesos iguales significa volver a 12.5% por ETF en cada rebalanceo, no mantener 12.5% todos los días.

El máximo de 15% se exigirá sobre los pesos objetivo al rebalancear. La deriva posterior puede superar ese nivel sin activar operaciones adicionales. Tras el último rebalanceo, mantendremos las posiciones hasta el corte disponible, aunque el periodo final sea parcial.

**Siguiente pregunta:** ¿Podemos reconstruir estas fechas y sus muestras admisibles sin utilizar información futura?

## 4. ¿Qué fechas podemos usar para rebalancear?

Construimos los cierres mensuales con el calendario publicado y, para cada uno, identificamos la muestra expansiva disponible hasta la sesión anterior. Ambos modelos optimizados compartirán esas filas. El último mes del corte, octubre de 2026, es parcial y queda en la auditoría como no elegible. El horizonte de riesgo es una sesión y la semivida, 1,102 sesiones.

El raw VIX se verifica con el hash de su captura y la alineación con el hash de su manifiesto. Los hashes de los dos Parquet de matrices se comparan con los registrados posteriormente al crear la alineación: el manifiesto original del corte no ofrece hashes históricos de referencia para ellos. El catálogo tampoco tiene aquí un hash de referencia; verificamos sus identificadores y estados. La disponibilidad histórica del VIX sigue siendo una convención retrospectiva, no una comprobación point-in-time.

In [1]:
from pathlib import Path
from datetime import date
import hashlib, json, site
import numpy as np

def project_root():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'scenario-portfolio-lab', parent / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if (candidate / 'config' / 'universe.csv').is_file() and (candidate / 'src' / 'scenario_portfolio_lab').is_dir():
                return candidate
    raise RuntimeError('Ejecute la notebook dentro del repositorio completo scenario-portfolio-lab.')

ROOT = project_root()
project_site = ROOT / '.venv' / 'Lib' / 'site-packages'
if project_site.is_dir():
    site.addsitedir(str(project_site))
import polars as pl

RUN = ROOT / 'data/published/cut_2026-10-09_run_2026-10-11_full_window_final'
ALIGNMENT = ROOT / 'data/published/vix_state_alignment_2026-10-31'
VIX_CAPTURE = ROOT / 'data/raw/vix_index_access_2026-10-29'
price_path = RUN / 'gold/adjusted_close_matrix.parquet'
return_path = RUN / 'gold/eligible_simple_total_return_matrix.parquet'
catalog_path = RUN / 'reports/eligible_universe.csv'
audit_path = ALIGNMENT / 'vix_return_alignment.parquet'
vix_path = VIX_CAPTURE / 'raw/VIX_INDEX_DATA_daily.json'
run_manifest = json.loads((RUN / 'run_manifest.json').read_text(encoding='utf-8'))
alignment_manifest = json.loads((ALIGNMENT / 'alignment_manifest.json').read_text(encoding='utf-8'))
vix_manifest = json.loads((VIX_CAPTURE / 'manifest.json').read_text(encoding='utf-8'))
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
integrity = {
    'VIX raw: hash de captura': sha256(vix_path) == vix_manifest['raw_sha256'] == alignment_manifest['inputs']['vix_raw']['sha256_capture_manifest_reference'],
    'Alineación: hash de su manifiesto': sha256(audit_path) == alignment_manifest['outputs']['alignment_table']['sha256'],
    'Precios: hash posterior de alineación': sha256(price_path) == alignment_manifest['inputs']['price_calendar']['sha256_computed_now'],
    'Retornos: hash posterior de alineación': sha256(return_path) == alignment_manifest['inputs']['simple_return_matrix']['sha256_computed_now'],
}
if not all(integrity.values()):
    raise AssertionError([name for name, ok in integrity.items() if not ok])

# Validamos los esquemas reales antes de seleccionar las columnas de trabajo.
price_schema = pl.read_parquet_schema(price_path); return_schema = pl.read_parquet_schema(return_path)
audit_schema = pl.read_parquet_schema(audit_path)
catalog = pl.read_csv(catalog_path)
required_audit = {'return_row_id', 'return_date', 'return_start_date', 'vix_source_date', 'vix_close', 'alignment_status'}
if list(price_schema)[0] != 'session_date' or list(return_schema)[0] != 'session_date' or not required_audit.issubset(audit_schema) or not {'symbol', 'asset_class', 'universe_status'}.issubset(catalog.columns):
    raise AssertionError('Esquema de entrada inesperado; revisar archivos de origen.')
eligible = catalog.filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible['symbol'].to_list()
SYMBOLS = ['SPY', 'VEA', 'EEM', 'TLT', 'BIL', 'GLD', 'USO', 'VNQ']
if len(eligible_symbols) != 125 or len(set(eligible_symbols)) != 125 or list(return_schema)[1:] != eligible_symbols:
    raise AssertionError('El catálogo elegible no coincide exactamente con las columnas de retornos.')
if any(eligible_symbols.count(symbol) != 1 or symbol not in price_schema for symbol in SYMBOLS):
    raise AssertionError('Falta o se repite un ETF en los insumos publicados.')
prices = pl.read_parquet(price_path, columns=['session_date', *SYMBOLS])
returns = pl.read_parquet(return_path, columns=['session_date', *SYMBOLS])
audit = pl.read_parquet(audit_path)
calendar = prices['session_date'].to_list()
cut = date.fromisoformat(run_manifest['approved_period']['end'])
start = date.fromisoformat(run_manifest['approved_period']['start'])
raw_vix = json.loads(vix_path.read_text(encoding='utf-8'))
if raw_vix.get('symbol') != 'VIX' or raw_vix.get('interval') != 'daily':
    raise AssertionError('El raw no identifica la serie VIX diaria.')
vix_records = [(date.fromisoformat(row['date']), float(row['close'])) for row in raw_vix['data']]
if len({d for d, _ in vix_records}) != len(vix_records):
    raise AssertionError('El raw VIX contiene fechas duplicadas.')
# No usamos los registros VIX posteriores al corte, aunque se conservan en el raw original.
vix_by_date = {d: value for d, value in vix_records if d <= cut}
if len(calendar) != 2205 or calendar[0] != start or calendar[-1] != cut or returns['session_date'].to_list() != calendar:
    raise AssertionError('El calendario de matrices no coincide con la ventana publicada.')
if calendar != sorted(calendar) or len(set(calendar)) != len(calendar):
    raise AssertionError('Las fechas del calendario no son únicas y crecientes.')
if returns.head(1).select(SYMBOLS).null_count().row(0) != (1,) * len(SYMBOLS):
    raise AssertionError('La primera fila de retornos debe ser nula por ausencia de precio anterior.')
return_values = returns.slice(1).select(SYMBOLS).to_numpy()
price_values = prices.select(SYMBOLS).to_numpy()
if not np.isfinite(return_values).all() or np.any(return_values <= -1) or not np.isfinite(price_values).all() or np.any(price_values <= 0):
    raise AssertionError('Hay retornos o precios inválidos en los ocho ETF.')

audit_rows = audit.to_dicts()
if len(audit_rows) != 2204 or [row['return_date'] for row in audit_rows] != calendar[1:]:
    raise AssertionError('La auditoría no conserva todas las fechas de retorno en orden.')
if audit_rows[0]['alignment_status'] != 'UNALIGNABLE_PRE_PUBLISHED_CALENDAR_START' or any(row['alignment_status'] != 'ALIGNED' for row in audit_rows[1:]):
    raise AssertionError('Estados de alineación inesperados; no se descartan filas silenciosamente.')
for i, row in enumerate(audit_rows):
    if row['return_row_id'] != i or row['return_start_date'] != calendar[i]:
        raise AssertionError(f'Identidad o inicio de retorno incorrecto: {row}')
    if i >= 1 and (row['vix_source_date'] != calendar[i-1] or row['vix_source_date'] not in vix_by_date or row['vix_close'] != vix_by_date[row['vix_source_date']] or not np.isfinite(row['vix_close']) or row['vix_close'] <= 0):
        raise AssertionError(f'Alineación histórica VIX x-2 inválida: {row}')
aligned_rows = audit_rows[1:]
aligned_dates = [row['return_date'] for row in aligned_rows]

# Un mes queda completo sólo si el calendario publicado alcanza el mes siguiente; octubre final es parcial.
month_ends = [i for i, day in enumerate(calendar) if i == len(calendar)-1 or (calendar[i+1].year, calendar[i+1].month) != (day.year, day.month)]
M, TAU, RISK_HORIZON = 504, 1102, 1
rebalance_rows = []
for i in month_ends:
    execution = calendar[i]; previous = calendar[i-1] if i > 0 else None
    fit = [d for d in aligned_dates if previous is not None and d <= previous]
    decision_vix = vix_by_date.get(previous)
    next_return = calendar[i+1] if i+1 < len(calendar) else None
    reasons = []
    if i == len(calendar)-1: reasons.append('MES_FINAL_PARCIAL')
    if len(fit) < M: reasons.append('HISTORIA_ALINEADA_INSUFICIENTE')
    if decision_vix is None or not np.isfinite(decision_vix) or decision_vix <= 0: reasons.append('VIX_DECISIÓN_NO_DISPONIBLE')
    if next_return is None: reasons.append('SIN_SESIÓN_POSTERIOR_EN_EL_CORTE')
    rebalance_rows.append({'fecha_ejecución': execution, 'fin_información': previous, 'inicio_ajuste': fit[0] if fit else None, 'fin_ajuste': fit[-1] if fit else None, 'observaciones': len(fit), 'fecha_vix_decisión': previous, 'vix_decisión': decision_vix, 'primer_retorno_nuevo': next_return, 'elegible': not reasons, 'motivo': ' | '.join(reasons) if reasons else 'ELEGIBLE'})
rebalance_audit = pl.DataFrame(rebalance_rows)
eligible_rebalances = rebalance_audit.filter(pl.col('elegible'))

# Comprobamos identidad, fechas y ventanas mediante posiciones, sin imputar ni ajustar filas.
positions = {day: i for i, day in enumerate(calendar)}
eligible_rows = eligible_rebalances.to_dicts()
reason_counts = {}
for row in rebalance_rows:
    if not row['elegible']:
        for reason in row['motivo'].split(' | '): reason_counts[reason] = reason_counts.get(reason, 0) + 1
controls = {**integrity,
    'Fechas únicas y crecientes': calendar == sorted(set(calendar)) and aligned_dates == sorted(set(aligned_dates)),
    'Ocho ETF en orden y elegibles': returns.columns == ['session_date', *SYMBOLS] and prices.columns == ['session_date', *SYMBOLS] and all(eligible_symbols.count(s) == 1 for s in SYMBOLS),
    'Cierres completos; octubre parcial excluido': all(positions[r['fecha_ejecución']] in month_ends[:-1] for r in eligible_rows) and rebalance_rows[-1]['motivo'].startswith('MES_FINAL_PARCIAL') and (cut.year, cut.month) == (2026, 10),
    'Ajuste anterior y conteo directo': all(r['fin_ajuste'] <= r['fin_información'] < r['fecha_ejecución'] and r['observaciones'] == sum(d <= r['fin_información'] for d in aligned_dates) for r in eligible_rows),
    'Mínimo y ventanas expansivas anidadas': all(r['observaciones'] >= M and r['inicio_ajuste'] == aligned_dates[0] for r in eligible_rows) and all(a['observaciones'] < b['observaciones'] for a, b in zip(eligible_rows, eligible_rows[1:])),
    'Alineación histórica x-2 y raw exacto': len(aligned_rows) == 2203 and all(r['vix_source_date'] == calendar[positions[r['return_date']]-2] and r['vix_close'] == vix_by_date[r['vix_source_date']] for r in aligned_rows),
    'VIX decisión en sesión anterior': all(r['fecha_vix_decisión'] == calendar[positions[r['fecha_ejecución']]-1] and r['vix_decisión'] == vix_by_date[r['fecha_vix_decisión']] for r in eligible_rows),
    'Primer retorno es sesión siguiente': all(r['primer_retorno_nuevo'] == calendar[positions[r['fecha_ejecución']]+1] and r['primer_retorno_nuevo'] > r['fecha_ejecución'] for r in eligible_rows),
    'Sin insumos posteriores al corte': all(d <= cut for d in calendar) and all(r['vix_source_date'] <= cut for r in aligned_rows) and all(r['fin_información'] <= cut for r in eligible_rows),
}
if not all(controls.values()):
    raise AssertionError([name for name, ok in controls.items() if not ok])
display(pl.DataFrame({'símbolo': SYMBOLS, 'clase': [eligible.filter(pl.col('symbol') == s)['asset_class'][0] for s in SYMBOLS]}))
print(f'Sesiones: {len(calendar)}; retornos: {len(calendar)-1}; filas alineadas: {len(aligned_rows)}; ETF: {len(SYMBOLS)}; M={M}; semivida={TAU}; horizonte={RISK_HORIZON}.')
print(f'Rebalanceos elegibles: {eligible_rebalances.height}; primer y último grupo de cinco:')
display(eligible_rebalances.head(5)); display(eligible_rebalances.tail(5))
display(pl.DataFrame({'motivo': list(reason_counts), 'conteo': list(reason_counts.values())}))
display(pl.DataFrame({'control': list(controls), 'resultado': ['PASS' if ok else 'FAIL' for ok in controls.values()]}))


símbolo,clase
str,str
"""SPY""","""etf"""
"""VEA""","""etf"""
"""EEM""","""etf"""
"""TLT""","""etf"""
"""BIL""","""etf"""
"""GLD""","""etf"""
"""USO""","""etf"""
"""VNQ""","""etf"""


Sesiones: 2205; retornos: 2204; filas alineadas: 2203; ETF: 8; M=504; semivida=1102; horizonte=1.
Rebalanceos elegibles: 81; primer y último grupo de cinco:


fecha_ejecución,fin_información,inicio_ajuste,fin_ajuste,observaciones,fecha_vix_decisión,vix_decisión,primer_retorno_nuevo,elegible,motivo
date,date,date,date,i64,date,f64,date,bool,str
2020-01-31,2020-01-30,2018-01-04,2020-01-30,521,2020-01-30,15.49,2020-02-03,true,"""ELEGIBLE"""
2020-02-28,2020-02-27,2018-01-04,2020-02-27,540,2020-02-27,39.16,2020-03-02,true,"""ELEGIBLE"""
2020-03-31,2020-03-30,2018-01-04,2020-03-30,562,2020-03-30,57.08,2020-04-01,true,"""ELEGIBLE"""
2020-04-30,2020-04-29,2018-01-04,2020-04-29,583,2020-04-29,31.23,2020-05-01,true,"""ELEGIBLE"""
2020-05-29,2020-05-28,2018-01-04,2020-05-28,603,2020-05-28,28.59,2020-06-01,true,"""ELEGIBLE"""


fecha_ejecución,fin_información,inicio_ajuste,fin_ajuste,observaciones,fecha_vix_decisión,vix_decisión,primer_retorno_nuevo,elegible,motivo
date,date,date,date,i64,date,f64,date,bool,str
2026-05-29,2026-05-28,2018-01-04,2026-05-28,2110,2026-05-28,15.74,2026-06-01,true,"""ELEGIBLE"""
2026-06-30,2026-06-29,2018-01-04,2026-06-29,2131,2026-06-29,17.65,2026-07-01,true,"""ELEGIBLE"""
2026-07-31,2026-07-30,2018-01-04,2026-07-30,2153,2026-07-30,17.09,2026-08-03,true,"""ELEGIBLE"""
2026-08-31,2026-08-28,2018-01-04,2026-08-28,2174,2026-08-28,14.43,2026-09-01,true,"""ELEGIBLE"""
2026-09-30,2026-09-29,2018-01-04,2026-09-29,2195,2026-09-29,16.04,2026-10-01,true,"""ELEGIBLE"""


motivo,conteo
str,i64
"""HISTORIA_ALINEADA_INSUFICIENTE""",24
"""MES_FINAL_PARCIAL""",1
"""SIN_SESIÓN_POSTERIOR_EN_EL_COR…",1


control,resultado
str,str
"""VIX raw: hash de captura""","""PASS"""
"""Alineación: hash de su manifie…","""PASS"""
"""Precios: hash posterior de ali…","""PASS"""
"""Retornos: hash posterior de al…","""PASS"""
"""Fechas únicas y crecientes""","""PASS"""
…,…
"""Mínimo y ventanas expansivas a…","""PASS"""
"""Alineación histórica x-2 y raw…","""PASS"""
"""VIX decisión en sesión anterio…","""PASS"""
